In [ ]:
import os
import json
import math
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")

[INFO] Using device: cuda


1.  PrunableLinear – the custom learnable-gate layer

In [ ]:
class PrunableLinear(nn.Module):
    """
    A drop-in replacement for nn.Linear that attaches a learnable
    scalar gate to every single weight.

    During the forward pass:
        gates         = sigmoid(gate_scores)          ∈ (0, 1)
        pruned_weight = weight  ⊙  gates              element-wise product
        output        = input @ pruned_weight.T + bias

    During training the optimiser adjusts BOTH `weight` and
    `gate_scores`.  The L1 sparsity loss pushes gate values toward 0,
    effectively removing the corresponding weight from the computation.

    Parameters
    ----------
    in_features  : int
    out_features : int
    bias         : bool  (default True)
    init_gate    : float – initial value for gate_scores before sigmoid.
                   sigmoid(2.0) ≈ 0.88, so gates start nearly open.
    """

    def __init__(self, in_features: int, out_features: int,
                 bias: bool = True, init_gate: float = 0.0):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        # ── standard weight & bias (same init as nn.Linear) ──
        self.weight = nn.Parameter(
            torch.empty(out_features, in_features)
        )
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter("bias", None)

        # ── gate scores: same shape as weight ──
        # Stored as *pre-sigmoid* logits so that the gradient can push
        # them to −∞ (gate → 0, weight removed) or +∞ (gate → 1).
        self.gate_scores = nn.Parameter(
            torch.full((out_features, in_features), fill_value=init_gate)
        )

        self._reset_weight()

    def _reset_weight(self):
        """Kaiming uniform init – same as default nn.Linear."""
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in = self.in_features
            bound  = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(self.bias, -bound, bound)

    # ------------------------------------------------------------------
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Step 1 : gates         = sigmoid(gate_scores)
        Step 2 : pruned_weight = weight * gates          (element-wise)
        Step 3 : return        F.linear(x, pruned_weight, bias)

        Gradients flow through both weight and gate_scores automatically
        because all operations are differentiable PyTorch ops.
        """
        gates         = torch.sigmoid(self.gate_scores)   # ∈ (0, 1)
        pruned_weight = self.weight * gates                # element-wise
        return F.linear(x, pruned_weight, self.bias)

    # ------------------------------------------------------------------
    def get_gates(self) -> torch.Tensor:
        """Return current gate VALUES (after sigmoid), detached."""
        with torch.no_grad():
            return torch.sigmoid(self.gate_scores).detach()

    # ------------------------------------------------------------------
    def sparsity_loss(self) -> torch.Tensor:
        """
        L1 norm of the gate values for THIS layer.
        = sum of all sigmoid(gate_scores)
        """
        return torch.sigmoid(self.gate_scores).mean()

    # ------------------------------------------------------------------
    def extra_repr(self) -> str:
        return (f"in={self.in_features}, out={self.out_features}, "
                f"bias={self.bias is not None}")


2.  The full network

In [ ]:
class SelfPruningNet(nn.Module):
    """
    Four-layer MLP classifier for CIFAR-10.

    All linear transforms use PrunableLinear so every weight has a
    corresponding learnable gate that can be driven to zero.
    BatchNorm + ReLU + Dropout are used between hidden layers for
    stable training.
    """

    def __init__(self, dropout: float = 0.3):
        super().__init__()

        self.prunable_layers: list[PrunableLinear] = []   # convenience list

        def pl(in_f, out_f):
            layer = PrunableLinear(in_f, out_f)
            self.prunable_layers.append(layer)
            return layer

        self.net = nn.Sequential(
            nn.Flatten(),                          # 3×32×32 → 3072

            pl(3072, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            pl(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            pl(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            pl(256, 10),                           # 10 CIFAR-10 classes
        )

    # ------------------------------------------------------------------
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    # ------------------------------------------------------------------
    def total_sparsity_loss(self) -> torch.Tensor:
        """Sum of L1 gate norms over ALL prunable layers."""
        return sum(layer.sparsity_loss() for layer in self.prunable_layers)

    # ------------------------------------------------------------------
    def sparsity_level(self, threshold: float = 1e-2) -> float:
        """
        Fraction of gates whose value < threshold.
        A gate below the threshold is considered 'pruned'.
        """
        all_gates = torch.cat(
            [layer.get_gates().view(-1) for layer in self.prunable_layers]
        )
        pruned = (all_gates < threshold).float().mean().item()
        return pruned * 100.0   # return as percentage

    # ------------------------------------------------------------------
    def all_gate_values(self) -> np.ndarray:
        """Return all gate values as a flat numpy array."""
        gates = torch.cat(
            [layer.get_gates().view(-1) for layer in self.prunable_layers]
        )
        return gates.cpu().numpy()

    # ------------------------------------------------------------------
    def total_parameters(self) -> dict:
        weight_params = sum(
            p.numel() for name, p in self.named_parameters()
            if "gate" not in name
        )
        gate_params = sum(
            p.numel() for name, p in self.named_parameters()
            if "gate" in name
        )
        return {"weights": weight_params, "gates": gate_params,
                "total": weight_params + gate_params}


3.  Data loading

In [ ]:
def get_dataloaders(batch_size: int = 128, num_workers: int = 2):
    """
    Download CIFAR-10 (if not cached) and return train/test DataLoaders.

    Augmentation (train only):
        RandomCrop(32, padding=4)  – shifts image by up to 4 px
        RandomHorizontalFlip       – mirrors horizontally 50 % of the time

    Normalisation: ImageNet-style per-channel mean/std computed on
    CIFAR-10 training set.
    """
    MEAN = (0.4914, 0.4822, 0.4465)
    STD  = (0.2470, 0.2435, 0.2616)

    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])

    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])

    train_set = torchvision.datasets.CIFAR10(
        root="./data", train=True,  download=True, transform=train_transform)
    test_set  = torchvision.datasets.CIFAR10(
        root="./data", train=False, download=True, transform=test_transform)

    train_loader = DataLoader(train_set, batch_size=batch_size,
                              shuffle=True,  num_workers=num_workers,
                              pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=256,
                              shuffle=False, num_workers=num_workers,
                              pin_memory=True)
    return train_loader, test_loader

4.  Training loop

In [ ]:
def train_one_epoch(model, loader, optimizer, lambda_sparse, epoch, epochs):
    """
    Single training epoch.

    Returns
    -------
    avg_ce_loss    : float – average cross-entropy loss
    avg_total_loss : float – average total loss (CE + λ·sparsity)
    accuracy       : float – training accuracy in %
    """
    model.train()
    ce_criterion = nn.CrossEntropyLoss()

    total_ce    = 0.0
    total_loss  = 0.0
    correct     = 0
    n_samples   = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()

        logits = model(images)

        # ── classification loss ──
        ce_loss = ce_criterion(logits, labels)

        # ── sparsity regularisation ──
        sparsity = model.total_sparsity_loss()

        # ── combined loss ──
        loss = ce_loss + lambda_sparse * sparsity

        loss.backward()

        # Clip gradients to prevent exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # ── bookkeeping ──
        batch_size = images.size(0)
        total_ce   += ce_loss.item()   * batch_size
        total_loss += loss.item()      * batch_size
        correct    += (logits.argmax(dim=1) == labels).sum().item()
        n_samples  += batch_size

    return (total_ce   / n_samples,
            total_loss / n_samples,
            100.0 * correct / n_samples)

5.  Evaluation loop

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    """Return test accuracy (%) over the full loader."""
    model.eval()
    correct   = 0
    n_samples = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        preds = model(images).argmax(dim=1)
        correct   += (preds == labels).sum().item()
        n_samples += images.size(0)
    return 100.0 * correct / n_samples

6.  Full training run for one lambda value

In [ ]:
def run_experiment(lambda_sparse: float,
                   train_loader,
                   test_loader,
                   epochs: int = 30,
                   lr: float = 1e-3,
                   weight_decay: float = 1e-4) -> dict:
    """
    Train a fresh SelfPruningNet with the given lambda_sparse.

    Returns a dict with:
        lambda        : float
        test_accuracy : float  (%)
        sparsity      : float  (%)
        gate_values   : np.ndarray  (all gate values, flattened)
        history       : dict  (per-epoch losses and accuracies)
    """
    print(f"\n{'='*60}")
    print(f"  λ = {lambda_sparse}")
    print(f"{'='*60}")

    model = SelfPruningNet(dropout=0.3).to(DEVICE)

    param_info = model.total_parameters()
    print(f"  Parameters → weights: {param_info['weights']:,} | "
          f"gates: {param_info['gates']:,} | "
          f"total: {param_info['total']:,}")

    optimizer = optim.Adam(model.parameters(), lr=lr,
                           weight_decay=weight_decay)

    # Cosine annealing LR schedule – reduces LR smoothly over training
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-5
    )

    history = {
        "ce_loss": [], "total_loss": [],
        "train_acc": [], "test_acc": [], "sparsity": []
    }

    best_test_acc = 0.0
    best_state    = None

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        ce_loss, total_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, lambda_sparse, epoch, epochs
        )

        test_acc = evaluate(model, test_loader)
        sparsity = model.sparsity_level()

        scheduler.step()

        history["ce_loss"].append(ce_loss)
        history["total_loss"].append(total_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)
        history["sparsity"].append(sparsity)

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_state    = {k: v.cpu().clone()
                             for k, v in model.state_dict().items()}

        elapsed = time.time() - t0
        print(f"  Epoch {epoch:3d}/{epochs} | "
              f"CE: {ce_loss:.4f} | "
              f"Total: {total_loss:.4f} | "
              f"Train: {train_acc:5.2f}% | "
              f"Test: {test_acc:5.2f}% | "
              f"Sparse: {sparsity:5.2f}% | "
              f"{elapsed:.1f}s")

    # ── Reload best checkpoint ──
    model.load_state_dict(best_state)
    final_test_acc = evaluate(model, test_loader)
    final_sparsity = model.sparsity_level()
    gate_values    = model.all_gate_values()

    print(f"\n  ✓ Best Test Accuracy : {final_test_acc:.2f}%")
    print(f"  ✓ Final Sparsity     : {final_sparsity:.2f}%")

    return {
        "lambda"       : lambda_sparse,
        "test_accuracy": final_test_acc,
        "sparsity"     : final_sparsity,
        "gate_values"  : gate_values,
        "history"      : history,
    }

7.  Plotting

In [ ]:
def plot_gate_distribution(results: list[dict], output_path: str = "gate_distribution.png"):
    """
    For each lambda, plot a histogram of the final gate values.

    A well-pruned model shows:
      - A large spike near 0  → pruned weights
      - A smaller cluster > 0 → surviving (active) weights
    """
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5),
                             facecolor="#0d0d0d")
    fig.suptitle("Gate Value Distributions After Training",
                 color="white", fontsize=16, fontweight="bold", y=1.02)

    colors = ["#00e5ff", "#ff6b6b", "#69ff47"]

    for i, (res, ax, color) in enumerate(zip(results, axes, colors)):
        gates    = res["gate_values"]
        lam      = res["lambda"]
        acc      = res["test_accuracy"]
        sparsity = res["sparsity"]

        ax.set_facecolor("#1a1a2e")
        ax.hist(gates, bins=80, color=color, alpha=0.85,
                edgecolor="none", density=True)

        # Mark the pruning threshold
        ax.axvline(x=0.01, color="white", linestyle="--",
                   linewidth=1.2, alpha=0.7, label="Prune threshold (0.01)")

        ax.set_title(
            f"λ = {lam}\nAcc: {acc:.1f}%  |  Sparsity: {sparsity:.1f}%",
            color="white", fontsize=12
        )
        ax.set_xlabel("Gate Value  (sigmoid output)", color="#aaaaaa")
        ax.set_ylabel("Density",                       color="#aaaaaa")
        ax.tick_params(colors="#aaaaaa")
        for spine in ax.spines.values():
            spine.set_edgecolor("#333355")
        ax.legend(facecolor="#1a1a2e", labelcolor="white", fontsize=9)

        # Annotation: % of gates near zero
        near_zero = (gates < 0.01).mean() * 100
        ax.annotate(f"{near_zero:.1f}% gates < 0.01",
                    xy=(0.03, 0.88), xycoords="axes fraction",
                    color=color, fontsize=10, fontweight="bold")

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight",
                facecolor="#0d0d0d")
    print(f"\n[INFO] Gate distribution plot saved → {output_path}")


def plot_training_curves(results: list[dict], output_path: str = "training_curves.png"):
    """
    For each lambda, plot test accuracy and sparsity over epochs.
    """
    n    = len(results)
    fig, axes = plt.subplots(2, n, figsize=(6 * n, 9),
                             facecolor="#0d0d0d")
    fig.suptitle("Training Curves per λ",
                 color="white", fontsize=16, fontweight="bold")

    colors = ["#00e5ff", "#ff6b6b", "#69ff47"]

    for i, (res, color) in enumerate(zip(results, colors)):
        hist = res["history"]
        lam  = res["lambda"]
        epochs = list(range(1, len(hist["test_acc"]) + 1))

        # ── accuracy ──
        ax_acc = axes[0][i]
        ax_acc.set_facecolor("#1a1a2e")
        ax_acc.plot(epochs, hist["train_acc"], color=color,
                    alpha=0.5, linewidth=1.5, label="Train Acc")
        ax_acc.plot(epochs, hist["test_acc"],  color=color,
                    alpha=1.0, linewidth=2.0, label="Test Acc")
        ax_acc.set_title(f"λ = {lam} — Accuracy", color="white")
        ax_acc.set_xlabel("Epoch", color="#aaaaaa")
        ax_acc.set_ylabel("Accuracy (%)", color="#aaaaaa")
        ax_acc.tick_params(colors="#aaaaaa")
        ax_acc.legend(facecolor="#1a1a2e", labelcolor="white")
        for sp in ax_acc.spines.values():
            sp.set_edgecolor("#333355")

        # ── sparsity ──
        ax_sp = axes[1][i]
        ax_sp.set_facecolor("#1a1a2e")
        ax_sp.plot(epochs, hist["sparsity"], color=color,
                   linewidth=2.0)
        ax_sp.set_title(f"λ = {lam} — Sparsity Level (%)", color="white")
        ax_sp.set_xlabel("Epoch", color="#aaaaaa")
        ax_sp.set_ylabel("Sparsity (%)", color="#aaaaaa")
        ax_sp.tick_params(colors="#aaaaaa")
        for sp in ax_sp.spines.values():
            sp.set_edgecolor("#333355")

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight",
                facecolor="#0d0d0d")
    print(f"[INFO] Training curves plot saved → {output_path}")


8.  Markdown report generator

In [ ]:
def save_markdown_report(results: list[dict],
                         output_path: str = "results_table.md"):
    """
    Generate the written report required by the case study:
      1. Explanation of why L1 on sigmoid gates encourages sparsity
      2. Results table (Lambda | Test Accuracy | Sparsity Level)
      3. Reference to the gate distribution plot
    """

    lines = []

    lines.append("# Self-Pruning Neural Network — Results Report\n")
    lines.append("**Case Study** : Tredence Analytics – AI Engineering Intern\n")
    lines.append(f"**Device used**: {str(DEVICE).upper()}\n\n")

    # ── Section 1 – Theory ──
    lines.append("---\n")
    lines.append("## 1. Why Does an L1 Penalty on Sigmoid Gates Encourage Sparsity?\n\n")
    lines.append(
        "Each weight `w` in the network is multiplied by a gate value "
        "`g = sigmoid(s)`, where `s` is a learnable scalar stored as "
        "`gate_scores`. The gate lives in `(0, 1)`, so it can partially "
        "or fully suppress the weight without changing its sign.\n\n"
    )
    lines.append(
        "The sparsity loss added to cross-entropy is the **L1 norm of all "
        "gate values**: `SparsityLoss = Σ |g_i| = Σ g_i` (since gates are "
        "always positive). This creates a constant downward gradient on "
        "every gate: `∂SparsityLoss / ∂g_i = 1`, which translates to a "
        "gradient on `gate_scores` of `sigmoid'(s) = g(1-g)`. The "
        "optimiser perpetually tries to push **every** gate toward zero "
        "unless the classification loss resists.\n\n"
    )
    lines.append(
        "**Why L1 and not L2?** An L2 penalty (`Σ g²`) produces a "
        "gradient of `2g` — as a gate nears zero the gradient also "
        "shrinks, so gates asymptotically approach zero but rarely reach "
        "it. The L1 gradient is *constant* (always 1), giving each gate "
        "a steady push all the way to zero, which is why L1 regularisation "
        "is widely known to produce **exact zeros** (true sparsity) while "
        "L2 produces only *small* values.\n\n"
    )
    lines.append(
        "The hyperparameter **λ** controls the trade-off: a small λ lets "
        "the classification objective dominate (high accuracy, low "
        "sparsity); a large λ forces aggressive pruning (high sparsity, "
        "possible accuracy drop). The gate distribution plot shows this "
        "clearly: higher λ pushes more gates into the spike at 0.\n\n"
    )

    # ── Section 2 – Results table ──
    lines.append("---\n")
    lines.append("## 2. Results Summary\n\n")
    lines.append("| Lambda (λ) | Test Accuracy (%) | Sparsity Level (%) |\n")
    lines.append("|:----------:|:-----------------:|:------------------:|\n")
    for res in results:
        lines.append(
            f"| {res['lambda']:.4f} | "
            f"{res['test_accuracy']:.2f} | "
            f"{res['sparsity']:.2f} |\n"
        )
    lines.append("\n")

    lines.append(
        "> **Reading the table**: As λ increases, sparsity rises because "
        "the penalty more aggressively suppresses gates. Test accuracy "
        "typically falls for very high λ since too many weights are "
        "removed before the network converges.\n\n"
    )

    # ── Section 3 – Plots reference ──
    lines.append("---\n")
    lines.append("## 3. Gate Value Distribution\n\n")
    lines.append(
        "The histogram below (`gate_distribution.png`) shows the "
        "distribution of **all** gate values across every `PrunableLinear` "
        "layer after training. A successful result shows:\n\n"
        "- A **large spike near 0** – gates that have been driven to zero, "
        "meaning the corresponding weights are effectively pruned.\n"
        "- A **smaller cluster away from 0** – gates that survived, "
        "representing the most important connections.\n\n"
        "The higher the λ, the more weight is in the spike at 0.\n\n"
    )
    lines.append("![Gate Distribution](gate_distribution.png)\n\n")
    lines.append("![Training Curves](training_curves.png)\n\n")

    # ── Section 4 – Architecture ──
    lines.append("---\n")
    lines.append("## 4. Network Architecture\n\n")
    lines.append("```\n")
    lines.append("Input  (3 × 32 × 32)  → Flatten → 3072 neurons\n")
    lines.append("  PrunableLinear(3072 → 1024)  + BatchNorm + ReLU + Dropout(0.3)\n")
    lines.append("  PrunableLinear(1024 →  512)  + BatchNorm + ReLU + Dropout(0.3)\n")
    lines.append("  PrunableLinear( 512 →  256)  + BatchNorm + ReLU + Dropout(0.3)\n")
    lines.append("  PrunableLinear( 256 →   10)  → logits\n")
    lines.append("```\n\n")
    lines.append(
        "Every PrunableLinear layer attaches a `gate_scores` tensor "
        "(same shape as `weight`). Effective weight = `weight × sigmoid(gate_scores)`.\n\n"
    )

    # ── Section 5 – Training setup ──
    lines.append("---\n")
    lines.append("## 5. Training Setup\n\n")
    lines.append("| Hyperparameter | Value |\n")
    lines.append("|:--------------|:------|\n")
    lines.append("| Optimiser     | Adam  |\n")
    lines.append("| Learning Rate | 1e-3 with CosineAnnealingLR |\n")
    lines.append("| Weight Decay  | 1e-4  |\n")
    lines.append("| Epochs        | 30    |\n")
    lines.append("| Batch Size    | 128   |\n")
    lines.append("| Dropout       | 0.3   |\n")
    lines.append("| Pruning Threshold | 1e-2 |\n")
    lines.append("| Dataset       | CIFAR-10 (50k train / 10k test) |\n\n")

    with open(output_path, "w") as f:
        f.writelines(lines)

    print(f"[INFO] Markdown report saved → {output_path}")


9.  Main

In [ ]:
def main():
    # ── Hyperparameters ──
    EPOCHS     = 30
    BATCH_SIZE = 128
    LAMBDAS    = [0.1, 0.3, 0.6]

    print("\n" + "="*60)
    print("  SELF-PRUNING NEURAL NETWORK  –  CIFAR-10")
    print("="*60)

    train_loader, test_loader = get_dataloaders(
        batch_size=BATCH_SIZE, num_workers=2
    )
    print(f"\n[INFO] Train batches : {len(train_loader)}")
    print(f"[INFO] Test  batches : {len(test_loader)}")

    results = []
    for lam in LAMBDAS:
        res = run_experiment(
            lambda_sparse = lam,
            train_loader  = train_loader,
            test_loader   = test_loader,
            epochs        = EPOCHS,
        )
        results.append(res)

    # ── Summary ──
    print("\n" + "="*60)
    print("  FINAL RESULTS SUMMARY")
    print("="*60)
    print(f"  {'Lambda':<12} {'Test Acc (%)':>14} {'Sparsity (%)':>14}")
    print(f"  {'-'*12} {'-'*14} {'-'*14}")
    for r in results:
        print(f"  {r['lambda']:<12.4f} {r['test_accuracy']:>14.2f} "
              f"{r['sparsity']:>14.2f}")

    # ── Plots & Report ──
    plot_gate_distribution(results, "gate_distribution.png")
    plot_training_curves(results,   "training_curves.png")
    save_markdown_report(results,   "results_table.md")

    print("\n[DONE] All outputs saved.")
    print("  ├─ gate_distribution.png")
    print("  ├─ training_curves.png")
    print("  └─ results_table.md")


if __name__ == "__main__":
    main()


  SELF-PRUNING NEURAL NETWORK  –  CIFAR-10

[INFO] Train batches : 391
[INFO] Test  batches : 40

  λ = 0.1
  Parameters → weights: 3,809,034 | gates: 3,803,648 | total: 7,612,682
  Epoch   1/30 | CE: 1.8298 | Total: 2.0295 | Train: 33.50% | Test: 42.60% | Sparse:  0.00% | 20.4s
  Epoch   2/30 | CE: 1.6842 | Total: 1.8835 | Train: 39.27% | Test: 45.13% | Sparse:  0.00% | 19.1s
  Epoch   3/30 | CE: 1.6300 | Total: 1.8290 | Train: 40.87% | Test: 45.45% | Sparse:  0.00% | 21.1s
  Epoch   4/30 | CE: 1.6015 | Total: 1.8003 | Train: 42.10% | Test: 47.56% | Sparse:  0.00% | 20.6s
  Epoch   5/30 | CE: 1.5769 | Total: 1.7756 | Train: 43.18% | Test: 49.04% | Sparse:  0.00% | 19.2s
  Epoch   6/30 | CE: 1.5575 | Total: 1.7561 | Train: 43.87% | Test: 49.40% | Sparse:  0.00% | 20.2s
  Epoch   7/30 | CE: 1.5440 | Total: 1.7425 | Train: 44.31% | Test: 49.73% | Sparse:  0.00% | 19.4s
  Epoch   8/30 | CE: 1.5336 | Total: 1.7321 | Train: 44.78% | Test: 50.63% | Sparse:  0.00% | 20.9s
  Epoch   9/30 | CE

In [ ]:
import shutil
import os

In [ ]:
dest = r"C:\Users\Dharani\OneDrive\Documents\Case study"
os.makedirs(dest, exist_ok=True)

In [ ]:
import os
print(os.getcwd())


/content


In [ ]:
import glob
files = glob.glob("**/*.png", recursive=True) + glob.glob("**/*.md", recursive=True)
for f in files:
    print(f)

pruning_results/gate_distribution.png
pruning_results/training_curves.png
pruning_results/results_table.md
sample_data/README.md


In [ ]:
dest = r"C:\Users\Dharani\OneDrive\Documents\Case study\pruning_results"
os.makedirs(dest, exist_ok=True)

In [ ]:
shutil.copy("pruning_results/gate_distribution.png", dest)
shutil.copy("pruning_results/training_curves.png",   dest)
shutil.copy("pruning_results/results_table.md",      dest)

'C:\\Users\\Dharani\\OneDrive\\Documents\\Case study\\pruning_results/results_table.md'

In [ ]:
print(os.path.abspath("pruning_results"))

/content/pruning_results


In [ ]:
from google.colab import files

In [ ]:
files.download("pruning_results/gate_distribution.png")
files.download("pruning_results/training_curves.png")
files.download("pruning_results/results_table.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>